# Computational Theory Problems

In [17]:
import numpy as np
import math
# Suppress overflow warnings - SHA-256 requires modulo 2^32 arithmetic
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning, message='overflow encountered in scalar add')
    

## Problem 1: Binary Words and Operations
----
### Parity - $x \oplus y \oplus z$ 
----
Used in rounds 20-39 & 64-79 of SHA-256 to operate on three 32-bit words to produce a 32-bit word output.
The output is the bitwise XOR of the three 32 bit "word" inputs.
Bitwise XOR meaning that each bit in the output is 1 if an odd number of the corresponding bits in the inputs are 1, and 0 otherwise.

Due to having 3 inputs, the parity function effectively counts the number of 1s in each bit position across the three inputs and sets the corresponding output bit to 1 if that count is odd, and to 0 if it is even.

e.g If we did a parity function that takes 2 bit inputs:<br>

> parity(00, 00, 01) would yield<br>
>- First bit: 0 + 0 + 0 = 0 (even) -> output 0<br>
>- Second bit: 0 + 0 + 1 = 1 (odd)  -> output 1<br>
>Resulting in output: 01<br>


In [18]:

def parity(x, y, z):
    """
    :param x: 32-bit word
    :param y: 32-bit word
    :param z: 32-bit word

    :return: 32-bit word representing the parity (XOR) of x, y, z

    Performs bitwise XOR on three 32-bit words.
    """
    x = np.uint32(x); y = np.uint32(y); z = np.uint32(z)
    return np.uint32(x ^ y ^ z)

def parityExamples():
    """
    Example cases demonstrating parity(x, y, z) == x ^ y ^ z (32-bit)
    """
    # Example cases
    examples = [
        ("all_zero",        np.uint32(0x00000000), np.uint32(0x00000000), np.uint32(0x00000000)),
        ("single_bit",      np.uint32(0x00000001), np.uint32(0x00000000), np.uint32(0x00000000)),
        ("two_same_bits",   np.uint32(0x00000001), np.uint32(0x00000001), np.uint32(0x00000000)),  # x ^ x == 0
        ("three_same_bits", np.uint32(0x00000001), np.uint32(0x00000001), np.uint32(0x00000001)),  # odd -> 1
        ("pattern_AAA",     np.uint32(0xAAAAAAAA), np.uint32(0x00000000), np.uint32(0x00000000)),
        ("pattern_mix",     np.uint32(0x12345678), np.uint32(0x9ABCDEF0), np.uint32(0x0F0F0F0F)),
    ]

    print("Parity function examples.")
    # Test parity function against expected XOR results 
    for name, a, b, c in examples:
        res = np.uint32(parity(a, b, c))
        expected = np.uint32(a ^ b ^ c)
        assert res == expected, f"parity mismatch for {name}"
        print(f"{name}: a=0x{int(a):08x} b=0x{int(b):08x} c=0x{int(c):08x} -> parity=0x{int(res):08x} {int(res):032b}")
    print()

parityExamples()

Parity function examples.
all_zero: a=0x00000000 b=0x00000000 c=0x00000000 -> parity=0x00000000 00000000000000000000000000000000
single_bit: a=0x00000001 b=0x00000000 c=0x00000000 -> parity=0x00000001 00000000000000000000000000000001
two_same_bits: a=0x00000001 b=0x00000001 c=0x00000000 -> parity=0x00000000 00000000000000000000000000000000
three_same_bits: a=0x00000001 b=0x00000001 c=0x00000001 -> parity=0x00000001 00000000000000000000000000000001
pattern_AAA: a=0xaaaaaaaa b=0x00000000 c=0x00000000 -> parity=0xaaaaaaaa 10101010101010101010101010101010
pattern_mix: a=0x12345678 b=0x9abcdef0 c=0x0f0f0f0f -> parity=0x87878787 10000111100001111000011110000111



----
### Choose - $(x \land y) \oplus (\neg x \land z)$
----

This function uses x as a mask to select bits from y where x is 1, and from z where x is 0.
Used in rounds 0-19 of SHA-256.<br>
By performing bitwise operations, it effectively implements:<br>
>result = (x AND y) XOR (NOT x AND z)

e.g For a "2-bit choose"<br>
>ch(01, 10, 11) = 10<br>
>x = 01<br> 
>y = 1**0** x1 = 0 -> z1 = 1 <br>
>z = **1**1 x2 = 1 -> y2 = 0 <br>
>r = **10**

|       | Bit 1 | Bit 2 |
| ----- |:-----:| -----:|
|   X   |  0    |   1   |
|   Y   |  1    | **0** |
|   Z   | **1** |   1   |
|   r   | **1** | **0** |


In [19]:
def ch(x, y, z):
    """
    :param x: 32-bit word
    :param y: 32-bit word
    :param z: 32-bit word

    :return: 32-bit word 

    Chooses bits from y and z based on x.
    """
    x = np.uint32(x); y = np.uint32(y); z = np.uint32(z)
    return np.uint32((x & y) | (np.uint32(~x) & z))

def chooseExamples():
    """
        Example cases demonstrating ch(x, y, z)
    """
    # Example cases
    examples = [
        ("all_zero",        np.uint32(0x00000000), np.uint32(0x00000000), np.uint32(0x00000000)),
        ("x_all_ones",      np.uint32(0xFFFFFFFF), np.uint32(0x12345678), np.uint32(0x9ABCDEF0)),
        ("x_all_zeros",     np.uint32(0x00000000), np.uint32(0x12345678), np.uint32(0x9ABCDEF0)),
        ("mixed_bits",      np.uint32(0xF0F0F0F0), np.uint32(0xAAAAAAAA), np.uint32(0x55555555)),
    ]

    print("Choose function examples.")
    # Test ch function against expected results
    for name, a, b, c in examples:
        res = np.uint32(ch(a, b, c))
        expected = np.uint32((a & b) | (~a & c))
        assert res == expected, f"ch mismatch for {name}"
        print(f"{name}: x=0x{int(a):08x} y=0x{int(b):08x} z=0x{int(c):08x} -> ch=0x{int(res):08x} {int(res):032b}")
    print()


chooseExamples()

Choose function examples.
all_zero: x=0x00000000 y=0x00000000 z=0x00000000 -> ch=0x00000000 00000000000000000000000000000000
x_all_ones: x=0xffffffff y=0x12345678 z=0x9abcdef0 -> ch=0x12345678 00010010001101000101011001111000
x_all_zeros: x=0x00000000 y=0x12345678 z=0x9abcdef0 -> ch=0x9abcdef0 10011010101111001101111011110000
mixed_bits: x=0xf0f0f0f0 y=0xaaaaaaaa z=0x55555555 -> ch=0xa5a5a5a5 10100101101001011010010110100101



----
### Majority - $(x \land y) \oplus (x \land z) \oplus (y \land z)$
----

Majority function: for each bit position, takes the majority value among x, y, z.

E.g For a "2-bit majority"<br>
>maj(01, 10, 11) = 11<br>
>x = 01<br> 
>y = 1**0** x1 = 0 -> z1 = 1 <br>
>y = 10<br> 
>x = 0**1** z0 = 0 -> y0 = 1<br>
>-----------------------------<br>
>result = 11 

|       | Bit 1 | Bit 2 |
| ----- |:-----:| -----:|
|   X   |  0    | **1** |
|   Y   | **1** |   0   |
|   Z   | **1** | **1** |
|   r   | **1** | **1** |

In [20]:
def maj(x, y, z):
    """
    :param x: 32-bit word
    :param y: 32-bit word
    :param z: 32-bit word

    :return: 32-bit word

    Majority function: for each bit position, the output bit is the majority value among the three input bits.
    """
    x = np.uint32(x); y = np.uint32(y); z = np.uint32(z)
    return np.uint32((x & y) | (x & z) | (y & z))

def majExamples():
    """
    Example cases demonstrating maj(x, y, z)
    """
    # Example cases
    examples = [
        ("all_zero",        np.uint32(0x00000000), np.uint32(0x00000000), np.uint32(0x00000000)),
        ("all_ones",        np.uint32(0xFFFFFFFF), np.uint32(0xFFFFFFFF), np.uint32(0xFFFFFFFF)),
        ("two_ones_one_zero", np.uint32(0xFFFFFFFF), np.uint32(0xFFFFFFFF), np.uint32(0x00000000)),
        ("two_zeros_one_one", np.uint32(0x00000000), np.uint32(0x00000000), np.uint32(0xFFFFFFFF)),
        ("mixed_bits",      np.uint32(0xF0F0F0F0), np.uint32(0xAAAAAAAA), np.uint32(0x55555555)),
    ]

    print("Majority function examples.")
    # Test maj function against expected results
    for name, a, b, c in examples:
        res = np.uint32(maj(a, b, c))
        expected = np.uint32((a & b) | (a & c) | (b & c))
        assert res == expected, f"maj mismatch for {name}"
        print(f"{name}: x=0x{int(a):08x} y=0x{int(b):08x} z=0x{int(c):08x} -> maj=0x{int(res):08x} {int(res):032b}")
    print()

majExamples()

Majority function examples.
all_zero: x=0x00000000 y=0x00000000 z=0x00000000 -> maj=0x00000000 00000000000000000000000000000000
all_ones: x=0xffffffff y=0xffffffff z=0xffffffff -> maj=0xffffffff 11111111111111111111111111111111
two_ones_one_zero: x=0xffffffff y=0xffffffff z=0x00000000 -> maj=0xffffffff 11111111111111111111111111111111
two_zeros_one_one: x=0x00000000 y=0x00000000 z=0xffffffff -> maj=0x00000000 00000000000000000000000000000000
mixed_bits: x=0xf0f0f0f0 y=0xaaaaaaaa z=0x55555555 -> maj=0xf0f0f0f0 11110000111100001111000011110000



----
### Σ0 - $ROTR^2(x) \oplus ROTR^{13}(x) \oplus ROTR^{22}(x)$
----
One of the six logical functions used in SHA-256.

Applies and returns the result of the following bitwise operations to the input x:
Exclusive OR (XOR) of:
- x rotated right by 2 bits
- x rotated right by 13 bits
- x rotated right by 22 bits

Rotations ensure that bits shifted out on one end are reintroduced on the other end.

$ROTR n(x)=(x >> n)| (x << w - n)$

$<<$ Left-shift operation, where x << n is obtained by discarding the left-most n
bits of the word x and then padding the result with n zeroes on the right.

$>>$ Right-shift operation, where x >> n is obtained by discarding the right-
most n bits of the word x and then padding the result with n zeroes on the
left.

Example
>Take last 2 bits: 10<br>
>Shift right by 2:  00110101<br>
>Add last 2 bits to front: **10**110101<br>
>Result: 10110101<br>

In [21]:
def Sigma0(x):
    """
    :param x: 32-bit word
    :return: 32-bit word

    Performs the Sigma0 of the SHA-256 algorithm on the input word.
    """
    x = np.uint32(x)
    res = (np.uint32(x >> 2) | np.uint32(x << np.uint32(32 - 2))) ^ (np.uint32(x >> 13) | np.uint32(x << np.uint32(32 - 13))) ^ (np.uint32(x >> 22) | np.uint32(x << np.uint32(32 - 22)))
    return np.uint32(res)

def Sigma0Examples():
    """
    Example cases demonstrating Sigma0(x)
    """
    # Example cases
    examples = [
        ("all_zero",        np.uint32(0x00000000)),
        ("all_ones",        np.uint32(0xFFFFFFFF)),
        ("single_bit",      np.uint32(0x00000001)),
        ("pattern_AAA",     np.uint32(0xAAAAAAAA)),
        ("pattern_555",     np.uint32(0x55555555)),
        ("mixed_bits",      np.uint32(0x12345678)),
    ]

    print("Sigma0 function examples.")
    # Test Sigma0 function against expected results
    for name, a in examples:
        res = np.uint32(Sigma0(a))
        # Manually compute expected result for verification
        expected = (np.uint32(a >> 2) | np.uint32(a << np.uint32(32 - 2))) ^ (np.uint32(a >> 13) | np.uint32(a << np.uint32(32 - 13))) ^ (np.uint32(a >> 22) | np.uint32(a << np.uint32(32 - 22)))
        assert res == expected, f"Sigma0 mismatch for {name}"
        print(f"{name}: x=0x{int(a):08x} -> Sigma0=0x{int(res):08x} {int(res):032b}")
    print()

Sigma0Examples()

Sigma0 function examples.
all_zero: x=0x00000000 -> Sigma0=0x00000000 00000000000000000000000000000000
all_ones: x=0xffffffff -> Sigma0=0xffffffff 11111111111111111111111111111111
single_bit: x=0x00000001 -> Sigma0=0x40080400 01000000000010000000010000000000
pattern_AAA: x=0xaaaaaaaa -> Sigma0=0x55555555 01010101010101010101010101010101
pattern_555: x=0x55555555 -> Sigma0=0xaaaaaaaa 10101010101010101010101010101010
mixed_bits: x=0x12345678 -> Sigma0=0x66146474 01100110000101000110010001110100



----
### Σ1 - $ROTR^6(x) \oplus ROTR^{11}(x) \oplus ROTR^{25}(x)$
----
One of the six logical functions used in SHA-256.

Applies and returns the result of the following bitwise operations to the input x: Exclusive OR (XOR) of:
- Right rotation by 6 bits
- Right rotation by 11 bits
- Right rotation by 25 bits
    
Rotation ensures that bits shifted out on one end are reintroduced on the other end.

See rotation example above under subheading for the Σ0 function.

In [22]:
def Sigma1(x):
    """
    :param x: 32-bit word
    :return: 32-bit word

    Performs the Sigma1 of the SHA-256 algorithm on the input word.
    """
    x = np.uint32(x)
    res = (np.uint32(x >> 6) | np.uint32(x << np.uint32(32 - 6))) ^ (np.uint32(x >> 11) | np.uint32(x << np.uint32(32 - 11))) ^ (np.uint32(x >> 25) | np.uint32(x << np.uint32(32 - 25)))
    return np.uint32(res)

----
### σ0 - $ROTR^{7}(x) \oplus ROTR^{18}(x) \oplus SHR^{3}(x)$
----
One of the six logical functions used in SHA-256.

Applies and returns the result of the following bitwise operations to the input x: Exclusive OR (XOR) of:
- Right rotation by 7 bits
- Right rotation by 18 bits
- Right shift by 3 bits

Rotation ensures that bits shifted out on one end are reintroduced on the other end. The logical right shift (SHR) discards low-order bits and inserts zeros at the high end.

See rotation example above under subheading for the Σ0 function.

Shifting differs from rotation in that bits shifted out are not reintroduced.
New bits are filled with zeros.

$SHR n(x)=x >> n$

In [23]:

def sigma0(x):
    """
    :param x: 32-bit word
    :return: 32-bit word

    Performs the sigma0 of the SHA-256 algorithm on the input word.
    """
    x = np.uint32(x)
    res = (np.uint32(x >> 7) | np.uint32(x << np.uint32(32 - 7))) ^ (np.uint32(x >> 18) | np.uint32(x << np.uint32(32 - 18))) ^ np.uint32(x >> 3)
    return np.uint32(res)

def sigma0Examples():
    """
    Example cases demonstrating sigma0(x)
    """
    # Example cases
    examples = [
        ("all_zero",        np.uint32(0x00000000)),
        ("all_ones",        np.uint32(0xFFFFFFFF)),
        ("single_bit",      np.uint32(0x00000001)),
        ("pattern_AAA",     np.uint32(0xAAAAAAAA)),
        ("pattern_555",     np.uint32(0x55555555)),
        ("mixed_bits",      np.uint32(0x12345678)),
    ]

    print("sigma0 function examples.")
    # Test sigma0 function against expected results
    for name, a in examples:
        res = np.uint32(sigma0(a))
        # Manually compute expected result for verification
        expected = (np.uint32(a >> 7) | np.uint32(a << np.uint32(32 - 7))) ^ (np.uint32(a >> 18) | np.uint32(a << np.uint32(32 - 18))) ^ np.uint32(a >> 3)
        assert res == expected, f"sigma0 mismatch for {name}"
        print(f"{name}: x=0x{int(a):08x} -> sigma0=0x{int(res):08x} {int(res):032b}")
    print()

sigma0Examples()

sigma0 function examples.
all_zero: x=0x00000000 -> sigma0=0x00000000 00000000000000000000000000000000
all_ones: x=0xffffffff -> sigma0=0x1fffffff 00011111111111111111111111111111
single_bit: x=0x00000001 -> sigma0=0x02004000 00000010000000000100000000000000
pattern_AAA: x=0xaaaaaaaa -> sigma0=0xeaaaaaaa 11101010101010101010101010101010
pattern_555: x=0x55555555 -> sigma0=0xf5555555 11110101010101010101010101010101
mixed_bits: x=0x12345678 -> sigma0=0xe7fce6ee 11100111111111001110011011101110



----
### σ1 - $ROTR^{17}(x) \oplus ROTR^{19}(x) \oplus SHR^{10}(x)$
----
One of the six logical functions used in SHA-256.

Applies and returns the result of the following bitwise operations to the input x: Exclusive OR (XOR) of:
- Right rotation by 17 bits
- Right rotation by 19 bits
- Right shift by 10 bits

In [24]:
def sigma1(x):
    """
    :param x: 32-bit word
    :return: 32-bit word

    Performs the sigma1 of the SHA-256 algorithm on the input word.
    """ 
    x = np.uint32(x)
    res = (np.uint32(x >> 17) | np.uint32(x << np.uint32(32 - 17))) ^ (np.uint32(x >> 19) | np.uint32(x << np.uint32(32 - 19))) ^ np.uint32(x >> 10)
    return np.uint32(res)

def sigma1Examples():
    """
    Example cases demonstrating sigma1(x)
    """
    # Example cases
    examples = [
        ("all_zero",        np.uint32(0x00000000)),
        ("all_ones",        np.uint32(0xFFFFFFFF)),
        ("single_bit",      np.uint32(0x00000001)),
        ("pattern_AAA",     np.uint32(0xAAAAAAAA)),
        ("pattern_555",     np.uint32(0x55555555)),
        ("mixed_bits",      np.uint32(0x12345678)),
    ]

    print("sigma1 function examples.")
    # Test sigma1 function against expected results
    for name, a in examples:
        res = np.uint32(sigma1(a))
        # Manually compute expected result for verification
        expected = (np.uint32(a >> 17) | np.uint32(a << np.uint32(32 - 17))) ^ (np.uint32(a >> 19) | np.uint32(a << np.uint32(32 - 19))) ^ np.uint32(a >> 10)
        assert res == expected, f"sigma1 mismatch for {name}"
        print(f"{name}: x=0x{int(a):08x} -> sigma1=0x{int(res):08x} {int(res):032b}")
    print()

sigma1Examples()

sigma1 function examples.
all_zero: x=0x00000000 -> sigma1=0x00000000 00000000000000000000000000000000
all_ones: x=0xffffffff -> sigma1=0x003fffff 00000000001111111111111111111111
single_bit: x=0x00000001 -> sigma1=0x0000a000 00000000000000001010000000000000
pattern_AAA: x=0xaaaaaaaa -> sigma1=0x002aaaaa 00000000001010101010101010101010
pattern_555: x=0x55555555 -> sigma1=0x00155555 00000000000101010101010101010101
mixed_bits: x=0x12345678 -> sigma1=0xa1f78649 10100001111101111000011001001001



## Problem 2: Fractional Parts of Cube Roots
----

We need to be able to find the cube roots of prime numbers, so we can generatethe 64 constant 32-bt words.

$K_0^{\{256\}}, K_1^{\{256\}},...,K_{63}^{\{256\}}$

These words represent the first thirty-two bits of the fractional parts of
the cube roots of the first sixty-four prime numbers.

**SHA-224** and **SHA-256** Constants in Hex form

`428a2f98 71374491 b5c0fbcf e9b5dba5 3956c25b 59f111f1 923f82a4 ab1c5ed5`<br>
`d807aa98 12835b01 243185be 550c7dc3 72be5d74 80deb1fe 9bdc06a7 c19bf174`<br>
`e49b69c1 efbe4786 0fc19dc6 240ca1cc 2de92c6f 4a7484aa 5cb0a9dc 76f988da`<br>
`983e5152 a831c66d b00327c8 bf597fc7 c6e00bf3 d5a79147 06ca6351 14292967`<br>
`27b70a85 2e1b2138 4d2c6dfc 53380d13 650a7354 766a0abb 81c2c92e 92722c85`<br>
`a2bfe8a1 a81a664b c24b8b70 c76c51a3 d192e819 d6990624 f40e3585 106aa070`<br>
`19a4c116 1e376c08 2748774c 34b0bcb5 391c0cb3 4ed8aa4a 5b9cca4f 682e6ff3`<br>
`748f82ee 78a5636f 84c87814 8cc70208 90befffa a4506ceb bef9a3f7 c67178f2`<br>


To do this, we need to be able to find and provide $n$ prime numebers.<br>
Then find the cube roots of those prime numbers.<br>
We'll then take the fractional part (the part after the decimel place) of the cube roots and convert them to hex.<br>

Doing the above for the first sixty-four prime numbers gives us the hex values above.





### **Step 1:** Finding Primes
----

To do this I will implement a function that uses the Sieve of Eratosthenes to find prime numbers until we have the required amount. 

#### **Sieve of Eratosthenes**
>Is an algorithm for finding all prime numbers upto any given limit.<br>
<br>
>So I will run the algorithm until the number of primes found matches the number wanted.<br>
<br>
>It does so by iteratively marking as not prime the multiples of each prime, starting with the first prime number, 2.<br>
<br>
>![Sieve of Eratosthenes Gif](https://upload.wikimedia.org/wikipedia/commons/9/94/Animation_Sieve_of_Eratosth.gif)

In [25]:
def primes(n):
    """
    :param n: integer
    :return: list of first n prime numbers

    Generates the first n prime numbers using the Sieve of Eratosthenes algorithm.
    """
    primes_list = []
    candidate = 2
    while len(primes_list) < n:
        is_prime = True
        for p in primes_list:
            if p * p > candidate:
                break
            if candidate % p == 0:
                is_prime = False
                break
        if is_prime:
            primes_list.append(candidate)
        candidate += 1
    return primes_list

def primeExamples():
    """
    Example cases demonstrating primes(n)
    """
    ns = [0, 1, 5, 10, 20, 64]
    print("Prime number generation examples.")
    for n in ns:
        prime_list = primes(n)
        print(f"First {n} primes: {prime_list}")
    print()

primeExamples()


Prime number generation examples.
First 0 primes: []
First 1 primes: [2]
First 5 primes: [2, 3, 5, 7, 11]
First 10 primes: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
First 20 primes: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71]
First 64 primes: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311]



### **Step 2:** Cube Roots
----
We get the cube roots by raising the prime to the power of 1/3.
Raising to a power of 1/3 is equivalent to taking the cube root.

Why does $x^{0.5} = \sqrt{x}$

$x^a * x^b = x^{a+b}$

This gives you...

$x^{0.5} * x^{0.5} = x^1$

or...

$(x^{0.5})2 = x$

In [26]:
def getPrimeRoots(primes):
    """
    :param primes: list of prime numbers
    :return: list of tuples containing prime roots

    """
    roots = []
    for prime in primes:
        # Compute cube root.
        root = prime ** (1/3)
        fractional_part = root - math.floor(root)
        first_32_bits = int(fractional_part * (2**32))
        roots.append((prime, first_32_bits))
    return roots

def primeRootExamples():
    """
    Example cases demonstrating getPrimeRoots(primes)
    """
    prime_list = primes(10)
    roots = getPrimeRoots(prime_list)
    print("Prime cube root fractional parts (first 32 bits):")
    for prime, root in roots:
        print(f"Prime: {prime}, Cube root fractional part (first 32 bits): 0x{root:08x}")
    print()

primeRootExamples()


Prime cube root fractional parts (first 32 bits):
Prime: 2, Cube root fractional part (first 32 bits): 0x428a2f98
Prime: 3, Cube root fractional part (first 32 bits): 0x71374491
Prime: 5, Cube root fractional part (first 32 bits): 0xb5c0fbcf
Prime: 7, Cube root fractional part (first 32 bits): 0xe9b5dba5
Prime: 11, Cube root fractional part (first 32 bits): 0x3956c25b
Prime: 13, Cube root fractional part (first 32 bits): 0x59f111f1
Prime: 17, Cube root fractional part (first 32 bits): 0x923f82a4
Prime: 19, Cube root fractional part (first 32 bits): 0xab1c5ed5
Prime: 23, Cube root fractional part (first 32 bits): 0xd807aa98
Prime: 29, Cube root fractional part (first 32 bits): 0x12835b01



### Step 3: Calculating - $K_0^{\{256\}}, K_1^{\{256\}},...,K_63^{\{256\}}$
----

We first find the first 64 primes and then take the first 32 bits of their cube root fractional parts.
Fractional parts meaning the digits after the decimel place.

We can verify the results against known values taken from the SHA-256 specification.


In [27]:

first_64_primes = primes(64)
prime_roots = getPrimeRoots(first_64_primes)

"""Check against known SHA-256 constants taken directly from the specification."""
known_sha256_constants = [
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2
]
# Extract computed constants from prime_roots
computed_constants = [root for prime, root in prime_roots]

# Find and print any mismatches between computed and known constants
mismatches = [
    (i, first_64_primes[i], computed_constants[i], known_sha256_constants[i])
    for i in range(min(len(computed_constants), len(known_sha256_constants)))
    if computed_constants[i] != known_sha256_constants[i]
]

if mismatches:
    print("Mismatches (index, prime, computed, known):")
    for i, p, comp, known in mismatches:
        print(f"{i}: prime={p} computed=0x{comp:08x} ({comp}) known=0x{known:08x} ({known})")
else:
    print("No mismatches found.")
    print()
    # Pretty-print computed SHA-256 constants as 8-per-line hex words
    hex_words = [f"0x{val:08x}" for val in computed_constants]
    print("Computed SHA-256 constants:")
    for i in range(0, len(hex_words), 8):
        print(" ".join(hex_words[i:i+8]))




No mismatches found.

Computed SHA-256 constants:
0x428a2f98 0x71374491 0xb5c0fbcf 0xe9b5dba5 0x3956c25b 0x59f111f1 0x923f82a4 0xab1c5ed5
0xd807aa98 0x12835b01 0x243185be 0x550c7dc3 0x72be5d74 0x80deb1fe 0x9bdc06a7 0xc19bf174
0xe49b69c1 0xefbe4786 0x0fc19dc6 0x240ca1cc 0x2de92c6f 0x4a7484aa 0x5cb0a9dc 0x76f988da
0x983e5152 0xa831c66d 0xb00327c8 0xbf597fc7 0xc6e00bf3 0xd5a79147 0x06ca6351 0x14292967
0x27b70a85 0x2e1b2138 0x4d2c6dfc 0x53380d13 0x650a7354 0x766a0abb 0x81c2c92e 0x92722c85
0xa2bfe8a1 0xa81a664b 0xc24b8b70 0xc76c51a3 0xd192e819 0xd6990624 0xf40e3585 0x106aa070
0x19a4c116 0x1e376c08 0x2748774c 0x34b0bcb5 0x391c0cb3 0x4ed8aa4a 0x5b9cca4f 0x682e6ff3
0x748f82ee 0x78a5636f 0x84c87814 0x8cc70208 0x90befffa 0xa4506ceb 0xbef9a3f7 0xc67178f2


## Problem 3: Padding
----

What is Padding in the SHS?


Read directly - "The purpose of this padding is to ensure that the padded message is a multiple of 512 or 1024
bits, depending on the algorithm."

### Why do we padd our messages?

For SHA-256, our message will be parsed in 512-bit blocks or 'words'.
We neeed to pad our messages to ensure they fit into these blocks properly.
By fitting properly, the SHA-256 algorithm can process the message in fixed-size chunks. 
<br>
Further, these 512-bit blocks are divided futher into 32 bit blocks denoted $M_1^{(i)}$ to $M_{15}^{(i)}$.

### How do we padd our messages?

Taken directly from the SHA-256 specification:
>"Suppose that the length of the message, $M$, is $l$ bits. 
<br>
>Append the bit “$1$” to the end of the message, followed by $k$ zero bits, where $k$ is the smallest, non-negative solution to the $l + 1 + k = 448mod512$. 
><br>Then append the 64-bit block that is equal to the number $l$ expressed using a binary representation. 
<br>
<br>
>For example, <br>
the (8-bit ASCII) message “**abc**” has length $8*3=24$,<br> 
so the message is padded with a one bit, <br>
then 448 -(24+1) = 423 zero bits, <br>
and then the message length, to become the 512-bit padded message"<br>

This essentially means:
1. Append a single '1' bit to the message.
2. Append '0' bits until the message length is congruent to 448 modulo 512, meaning the length is 64 bits off being a multiple of 512 (the remaining 64 bits are used to store the original message length in bits).
3. Append the original message length as a 64-bit big-endian integer.
4. The resulting padded message length will be a multiple of 512 bits.

<br>
<br>

>An interesting case here is when the message length is exactyl 512 bits (64 bytes).<br>
>Intuitively this message is already the correct block size, according to the SHA-256 padding rules.<br>
>But per the specification, we still need to add an additional block for padding and length encoding.<br>
>Due to the additional block needing to be exactlty 512 bits we'll end up with two 64 byte message blocks.<br>



In [28]:
def block_parse(msg):
    """
    Generator function that parses a message according to SHA-256 specification (sections 5.1.1 and 5.2.1).
    
    :param msg: bytes object to be parsed
    :yield: 512-bit (64-byte) blocks as bytes objects
    
    Implements message padding as per SHA-256 specification:
    1. Append bit '1' to the message
    2. Append '0' bits until length ≡ 448 (mod 512)
    3. Append 64-bit representation of original message length
    """
    # Get original message length in bits
    original_length_bits = len(msg) * 8
    
    # Step 1: Append the '1' bit (as byte 0x80 = 10000000 in binary)
    padded_msg = msg + b'\x80'
    
    # Step 2: Calculate number of zero bytes needed
    # We need length ≡ 448 (mod 512) bits, or 56 (mod 64) bytes
    # After appending 0x80, we need to reach 56 bytes mod 64, leaving 8 bytes for length
    current_length = len(padded_msg)
    zero_padding_length = (56 - current_length % 64) % 64
    padded_msg += b'\x00' * zero_padding_length
    
    # Step 3: Append original length as 64-bit big-endian integer
    padded_msg += original_length_bits.to_bytes(8, byteorder='big')
    
    # Yield 512-bit (64-byte) blocks
    for i in range(0, len(padded_msg), 64):
        yield padded_msg[i:i+64]


def block_parse_examples():
    """
    Test the block_parse generator with various message lengths
    """
    test_messages = [
        (b"", "empty message"),
        (b"abc", "3-byte message (24 bits)"),
        (b"a" * 55, "55-byte message (just under one block)"),
        (b"a" * 56, "56-byte message (needs two blocks)"),
        (b"a" * 64, "64-byte message (exactly one block)"),
        (b"The quick brown fox jumps over the lazy dog", "44-byte message"),
    ]
    
    print("Block parsing examples with padding:")
    print("=" * 80)
    
    for msg, description in test_messages:
        print(f"\nTest: {description}")
        print(f"Original length: {len(msg)} bytes ({len(msg) * 8} bits)")
        
        blocks = list(block_parse(msg))
        print(f"Number of 512-bit blocks: {len(blocks)}")
        
        for idx, block in enumerate(blocks):
            print(f"  Block {idx}: {len(block)} bytes")
            # Show first and last 16 bytes of each block
            if len(block) <= 32:
                print(f"    Content: {block.hex()}")
            else:
                print(f"    First 16 bytes: {block[:16].hex()}")
                print(f"    Last 16 bytes:  {block[-16:].hex()}")
            
            # For the last block, show the length encoding
            if idx == len(blocks) - 1:
                length_bytes = block[-8:]
                decoded_length = int.from_bytes(length_bytes, byteorder='big')
                print(f"    Encoded length: {decoded_length} bits (last 8 bytes: {length_bytes.hex()})")
        
        print("-" * 80)

block_parse_examples()

Block parsing examples with padding:

Test: empty message
Original length: 0 bytes (0 bits)
Number of 512-bit blocks: 1
  Block 0: 64 bytes
    First 16 bytes: 80000000000000000000000000000000
    Last 16 bytes:  00000000000000000000000000000000
    Encoded length: 0 bits (last 8 bytes: 0000000000000000)
--------------------------------------------------------------------------------

Test: 3-byte message (24 bits)
Original length: 3 bytes (24 bits)
Number of 512-bit blocks: 1
  Block 0: 64 bytes
    First 16 bytes: 61626380000000000000000000000000
    Last 16 bytes:  00000000000000000000000000000018
    Encoded length: 24 bits (last 8 bytes: 0000000000000018)
--------------------------------------------------------------------------------

Test: 55-byte message (just under one block)
Original length: 55 bytes (440 bits)
Number of 512-bit blocks: 1
  Block 0: 64 bytes
    First 16 bytes: 61616161616161616161616161616161
    Last 16 bytes:  616161616161618000000000000001b8
    Encoded l

## Problem 4: Hashes
---

Write a function ```hash(current, block)``` that calculates the next hash value given the current hash value <br>
and the next message block according to section 6.2.2 SHA-256 Hash Computation on page 22 of the Secure Hash Standard.

What is the message schedule ${\{W_t\}}$.

They are 32 bit chunks
$W_0, W_1, ..., W_62, w_63$

The first 16 or $ 0 \le t \le 15$ are $M_0, ..., M_15$

$ 16 \le t \le 63$ are given by iteratively by the formula $ σ_1^{\{256\}}(W_{t-2})+W_{t-7}+σ0(W_{t-15})+W_{T-16} $

### Initial Hash Values - $H_0^{(0)}, H_1^{(0)}, ..., H_7^{(0)}$
----

The initial hash values are the first 32 bits of the fractional parts of the **square roots** (not cube roots) of the first 8 primes (2, 3, 5, 7, 11, 13, 17, 19).

**Note:** This differs from the K constants which use **cube roots** of the first 64 primes.

These initial values are:
- $H_0^{(0)} = $ `0x6a09e667`
- $H_1^{(0)} = $ `0xbb67ae85`
- $H_2^{(0)} = $ `0x3c6ef372`
- $H_3^{(0)} = $ `0xa54ff53a`
- $H_4^{(0)} = $ `0x510e527f`
- $H_5^{(0)} = $ `0x9b05688c`
- $H_6^{(0)} = $ `0x1f83d9ab`
- $H_7^{(0)} = $ `0x5be0cd19`




---

#### For each round t from 0 to 63, perform the following operations:

1. **Calculate Σ₁(e)**: Apply the Σ₁ function to working variable e
   - `Σ₁(e) = ROTR⁶(e) ⊕ ROTR¹¹(e) ⊕ ROTR²⁵(e)`

2. **Calculate Ch(e, f, g)**: Apply the choose function
   - `Ch(e, f, g) = (e ∧ f) ⊕ (¬e ∧ g)`

3. **Calculate T₁**: First temporary word
   - `T₁ = h + Σ₁(e) + Ch(e, f, g) + Kₜ + Wₜ`
   - Where Kₜ is the t-th constant from the K array
   - Where Wₜ is the t-th word from the message schedule

4. **Calculate Σ₀(a)**: Apply the Σ₀ function to working variable a
   - `Σ₀(a) = ROTR²(a) ⊕ ROTR¹³(a) ⊕ ROTR²²(a)`

5. **Calculate Maj(a, b, c)**: Apply the majority function
   - `Maj(a, b, c) = (a ∧ b) ⊕ (a ∧ c) ⊕ (b ∧ c)`

6. **Calculate T₂**: Second temporary word
   - `T₂ = Σ₀(a) + Maj(a, b, c)`

7. **Update working variables**: Rotate and update the eight working variables
   - `h = g`
   - `g = f`
   - `f = e`
   - `e = d + T₁`
   - `d = c`
   - `c = b`
   - `b = a`
   - `a = T₁ + T₂`

All additions are performed modulo 2³².

After completing all 64 rounds, add the working variables to the current hash value to produce the intermediate/final hash.


    

In [32]:
def hash(current, block):
    """
    :param current: list of 8 32-bit words representing the current hash value
    :param block: bytes object of length 64 (512 bits) representing the next message block
    :return: list of 8 32-bit words representing the updated hash value
    """
    # Initialize working variables with current hash value
    a, b, c, d, e, f, g, h = current 
    # Prepare the message schedule array W
    W = [0] * 64
    # Break block into sixteen 32-bit big-endian words
    for i in range(16):
        W[i] = int.from_bytes(block[i*4:(i+1)*4], byteorder='big')
    # Extend the first 16 words into the remaining 48 words of the message schedule array
    for i in range(16, 64):
        s0 = sigma0(W[i - 15])
        s1 = sigma1(W[i - 2])
        W[i] = np.uint32(W[i - 16] + s0 + W[i - 7] + s1)
    # Initialize hash value for this chunk
    K = [root for prime, root in prime_roots]  # SHA-256 constants
    # Main loop
    for i in range(64):
        S1 = Sigma1(e)
        ch_e = ch(e, f, g)
        temp1 = np.uint32(h + S1 + ch_e + K[i] + W[i])
        S0 = Sigma0(a)
        maj_a = maj(a, b, c)
        temp2 = np.uint32(S0 + maj_a)
        
        h = g
        g = f
        f = e
        e = np.uint32(d + temp1)
        d = c
        c = b
        b = a
        a = np.uint32(temp1 + temp2)
    # Add the compressed chunk to the current hash value    
    new_hash = [
        np.uint32(current[0] + a),
        np.uint32(current[1] + b),
        np.uint32(current[2] + c),
        np.uint32(current[3] + d),
        np.uint32(current[4] + e),
        np.uint32(current[5] + f),
        np.uint32(current[6] + g),
        np.uint32(current[7] + h),
    ]
    return new_hash

def hashExamples():
    """
    Example demonstrating hash(current, block) for a single block.
    This shows the hash function processing the padded "abc" message.
    """
    # Initial hash values for SHA-256
    initial_hash = [
        0x6a09e667,
        0xbb67ae85,
        0x3c6ef372,
        0xa54ff53a,
        0x510e527f,
        0x9b05688c,
        0x1f83d9ab,
        0x5be0cd19,
    ]
    
    # Example: padded "abc" message (64 bytes)
    # 'abc' = 0x616263, then 0x80 padding bit, then zeros, then length (24 bits = 0x18)
    example_block = bytes.fromhex(
        '61626380000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000018'
    )
    
    print("Single block hash example (padded 'abc'):")
    print(f"Block content: {example_block.hex()}")
    print(f"Block length: {len(example_block)} bytes")
    
    updated_hash = hash(initial_hash, example_block)
    hash_hex = ''.join(f'{int(h):08x}' for h in updated_hash)
    
    print(f"\nFinal hash: {hash_hex}")
    print(f"Expected:   ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad")
    print(f"Match: {hash_hex == 'ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad'}")

hashExamples()

def sha256(msg):
    """
    Complete SHA-256 hash function.
    
    :param msg: bytes object to hash
    :return: 256-bit hash as a hex string
    
    Implements the full SHA-256 algorithm by:
    1. Padding the message using block_parse()
    2. Processing each 512-bit block through hash()
    3. Returning the final hash value
    """
    # Initial hash values (first 32 bits of fractional parts of square roots of first 8 primes)
    current_hash = [
        0x6a09e667,
        0xbb67ae85,
        0x3c6ef372,
        0xa54ff53a,
        0x510e527f,
        0x9b05688c,
        0x1f83d9ab,
        0x5be0cd19,
    ]
    
    # Process each block
    for block in block_parse(msg):
        current_hash = hash(current_hash, block)
    
    # Convert final hash to hex string
    return ''.join(f'{int(h):08x}' for h in current_hash)

def sha256_verification():
    """
    Verify SHA-256 implementation against known test vectors from the SHA-256 specification.
    Tests include empty string, single block, and multi-block messages.
    """
    import hashlib
    
    test_cases = [
        (b"", "empty string"),
        (b"abc", "single block message"),
        (b"abcdbcdecdefdefgefghfghighijhijkijkljklmklmnlmnomnopnopq", "two block message"),
        (b"The quick brown fox jumps over the lazy dog", "pangram"),
        (b"a" * 1000, "1000 'a' characters"),
    ]
    
    print("SHA-256 Verification Against Python's hashlib:")
    print("=" * 100)
    
    all_pass = True
    for msg, description in test_cases:
        # Your implementation
        my_hash = sha256(msg)
        # Python's hashlib for verification
        expected = hashlib.sha256(msg).hexdigest()
        
        match = my_hash == expected
        all_pass = all_pass and match
        symbol = "✓" if match else "✗"
        
        print(f"\n{symbol} Test: {description}")
        print(f"  Message length: {len(msg)} bytes")
        if len(msg) <= 50:
            print(f"  Message: {msg}")
        else:
            print(f"  Message: {msg[:47]}...")
        print(f"  Your hash:     {my_hash}")
        print(f"  Expected hash: {expected}")
        if not match:
            print(f"MISMATCH!")
    
    print("=" * 100)
    if all_pass:
        print("✓ All tests passed!")
    else:
        print("✗ Some tests failed!")
    
    return all_pass

sha256_verification()

Single block hash example (padded 'abc'):
Block content: 61626380000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000018
Block length: 64 bytes

Final hash: ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
Expected:   ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
Match: True
SHA-256 Verification Against Python's hashlib:

✓ Test: empty string
  Message length: 0 bytes
  Message: b''
  Your hash:     e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
  Expected hash: e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855

✓ Test: single block message
  Message length: 3 bytes
  Message: b'abc'
  Your hash:     ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
  Expected hash: ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad

✓ Test: two block message
  Message length: 56 bytes
  Message: b'abcdbcdecdefdefgefghfghighijhijkijkljklmklmnl

True

## Problem 5: Passwords

## End